### Part 1 - Basics

Does LIMIT 10 make queries (a) and (b) cheaper?

Query A 
- with limit: 135.145 ms
- without: 169.537 ms

Query B
- with limit: 183.365 ms
- without: 284.444 ms

It does make a difference, because grouping and filtering prepare the data for read, and limit tells the program to stop after the tenth row and not go through all rows.

### Part 2 - Activities


In [ ]:
DROP TABLE IF EXISTS orders_big;

CREATE TABLE orders_big (
  id SERIAL PRIMARY KEY,
  customer_name TEXT,
  product_catergory TEXT,
  quantity INTEGER,
  price_per_unit DEC,
  order_date DATE,
  country TEXT
);

\COPY orders_big(customer_name,
  product_catergory,
  quantity,
  price_per_unit,
  order_date,
  country) FROM '/data/orders_1M.csv' DELIMITER ',' CSV HEADER;


A. Wich order has the highest price per unit?
```
SELECT * FROM orders_big ORDER BY price_per_unit DESC LIMIT 10;
```
<ul>
    <li>841292 | Emma Brown | Automotive | 3 | 2000.0 | 2024-10-11 | Italy</li>
</ul>

B. What are the top 3 product categories with the highest total quantity sold across all orders?
```
SELECT
    product_catergory,
    SUM(quantity) AS total_quantity_sold
FROM orders_big
GROUP BY product_catergory
ORDER BY total_quantity_sold DESC
LIMIT 3;
```
<ul>
    <li>Health & Beauty | 300842</li>
    <li>Electronics     | 300804</li>
    <li>Toys            | 300598</li>
</ul>

C. What is the total revenue per product category?
```
SELECT
    product_catergory,
    SUM(price_per_unit * quantity) AS total_revenue
FROM orders_big
GROUP BY product_catergory
ORDER BY total_revenue DESC;
```
<ul>
    <li>Automotive | 306589798.86</li>
    <li>Electronics | 241525009.45</li>
    <li>Home & Garden | 78023780.09</li>
    <li>Sports | 61848990.83</li>
    <li>Health & Beauty | 46599817.89</li>
    <li>Office Supplies | 38276061.64</li>
</ul>

D. WHo are the top 5 customers by total spending?
```
SELECT
    customer_name,
    SUM(price_per_unit * quantity) AS total_spending
FROM orders_big
GROUP BY customer_name
ORDER BY total_spending DESC
LIMIT 5;
```
<ul>
    <li>Carol Taylor | 991179.18</li>
    <li>Nina Lopez | 975444.95</li>
    <li>Daniel Jackson | 959344.48</li>
    <li>Carol Lewis | 947708.57</li>
    <li>Daniel Young | 946030.14</li>
</ul>

E. Look at the spending totals in D - and at how many orders each of those customers has. What do you notice? Open ecommerce/dataset_generator.py and explain why the data looks like this.

```
SELECT
    customer_name,
    COUNT(*) AS order_count
FROM orders_big
WHERE customer_name IN (
    'Carol Taylor',
    'Nina Lopez',
    'Daniel Jackson',
    'Carol Lewis',
    'Daniel Young'
)
GROUP BY customer_name
ORDER BY order_count DESC;
```

Customer order count:
<ul>
    <li>Daniel Jackson | 1033</li>
    <li>Carol Taylor | 1028</li>
    <li>Nina Lopez | 980</li>
    <li>Daniel Young | 973</li>
    <li>Carol Lewis | 943</li>
</ul>

```
SELECT
    customer_name,
    COUNT(*) AS occurrences
FROM orders_big
WHERE customer_name IN (
    'Carol Taylor',
    'Nina Lopez',
    'Daniel Jackson',
    'Carol Lewis',
    'Daniel Young'
)
GROUP BY customer_name
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;
```

How often these names appear:
<ul>
    <li>Daniel Jackson | 1033</li>
    <li>Carol Taylor | 1028</li>
    <li>Nina Lopez | 980</li>
    <li>Daniel Young | 973</li>
    <li>Carol Lewis | 943</li>
</ul>

The code in the generator makes every customer oder a random amount of a random category, in our case the top five made the most orders of the most expensive products, also the names are in the database multiple times.

#### 2.2 - Why is this self-join so slow?

##### Step 1
```
SELECT COUNT(*)
FROM people_100k p1
JOIN people_100k p2
  ON p1.country = p2.country;
```
Time for:
<ul>
    <li>50k
        <ul>
            <li>878.942 ms</li>
        </ul>
    </li>
    <li>100k
        <ul>
            <li>3565.271 ms</li>
        </ul>
    </li>
    <li>200k
        <ul>
            <li>14523.855 ms</li>
        </ul>
    </li>
</ul>

The time grows around 4 times the previous one, while the row count doubles. 

* 50k: 878.942 ms

* 100k: 3 565.271 ms

* 200k: 14 523.855 ms

* 400k (estimated): 58 095.42 ms

* 800k (estimated): 232 381.68 ms

* 1 million (estimated): 363,096.38 ms (about 6 minutes and 3 seconds)


##### Step 2

```
EXPLAIN ANALYZE
SELECT COUNT(*)
FROM people_100k p1
JOIN people_100k p2
  ON p1.country = p2.country;
```

Execution Time for 100k: 7 361.316 ms

The index was used, but the query still is expensive because the join produces so many matching pairs. An index can make reading rows faster, but the main cost is processing the join results.

```
->  Parallel Index Only Scan using idx_people_100k_country on people_100k p1
->  Parallel Index Only Scan using idx_people_100k_country on people_100k p2

->  Parallel Hash Join  (cost=2332.11..469620.67 rows=64629045 width=0) (actual time=174.292..4325.974 rows=54973254.00 loops=2)
                     Hash Cond: (p1.country = p2.country)
```

##### Step 3

Since there are n^2 combinations, we can use:
```
SELECT SUM(country_count * country_count) AS total_pairs
FROM (
    SELECT country, COUNT(*) AS country_count
    FROM people_100k
    GROUP BY country
) AS country_counts;
```
109946508 vs. 109946508 (same)
2687.108 ms vs. 7.157 ms

Using this one people_big:
total pairs: 10983941260
Time: 211.893 ms

##### Step 4

The self-join creates k × k pairs for every country with k people. Double the rows means about 4× the work, so 1M rows is about 400× the work of 50k rows. The time goes into creating and counting billions of pairs.

The rewrite:
```sql
SELECT SUM(cnt * cnt) FROM (
  SELECT country, COUNT(*) AS cnt FROM people_big GROUP BY country
) t;
```
shows the same result, one scan, linear time.

- More hardware: only a constant speedup (10× machine = at most 10× faster), but 10× more data = 100× more work. A bigger machine helps only a little.
- A cluster: the data is split across several computers, for example by country, and each computer joins its own part at the same time. With 10 computers it could be up to 10× faster, but it also costs 10× more. The biggest country still takes the longest, moving data between computers over the network is slow, and the billions of result pairs still have to be stored somewhere.
- If the business really needs the pairs: the result is about 11 billion rows, so no method can be faster than creating them.

The Limits of OLTP (large-scale cloud environment)
- Row storage reads whole rows, not just country.
- Single node, hard to scale out.
- One long query slows down everyone's transactions.
- Bad queries could cost money